# Análisis de títulos de YouTube — versión solo español

**Archivo de datos:** el CSV más reciente `youtube_videos_largos_*.csv` generado por `Extraccion_YouTube.ipynb`

**Orden:** corre siempre las celdas 0 a 4b. Después, las celdas 5 a 11 son independientes y se pueden correr en cualquier orden (la 11 usa el mejor modelo de la 8 si ya la corriste; si no, usa C4). Usa GPU (Entorno de ejecución → Cambiar tipo → T4) para la celda 4.

| Celda | Qué hace |
|---|---|
| 0 | Subir el CSV |
| 1 | Revisar columnas y detectar la columna del canal |
| 2 | Idioma por canal y filtro **solo español** |
| 3 | Rasgos del título y variable objetivo (log10) |
| 4 | Sentimiento: VADER (línea base) y pysentimiento (español) |
| 4b | Configuración común de los experimentos |
| 5 | E1: estadísticos de vistas |
| 6 | E2: sentimiento vs. vistas |
| 7 | E3: dependencia con pruebas de permutación y Benjamini-Hochberg |
| 8 | Ablación C1–C4 repetida + prueba t corregida |
| 9 | CF1: 5 vs. 10 particiones |
| 10 | CF2: validación dejando un canal fuera y árboles regularizados |
| 10b | CF2b: título dentro del canal, antigüedad, sin el canal más grande y desempeño por canal |
| 11 | CF3: importancia por permutación, importancia agrupada y SHAP en datos de prueba |
| 12 | Resumen de resultados y descarga |
| 13 | H1: prueba de permutación (1000) con avance visible |


In [ ]:
# CELDA 0 — Subir CSV a Colab (córrela cada vez que se reinicie el entorno)
from google.colab import files
import os
uploaded = files.upload()
print([f for f in os.listdir(".") if f.endswith(".csv")])

In [ ]:
# CELDA 1 — Cargar y revisar columnas
!pip install -q vaderSentiment dcor emoji shap pysentimiento
import re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

import glob
INPUT_FILE  = sorted(glob.glob("youtube_videos_largos_*.csv"))[-1]   # la extracción más reciente que hayas subido
SEED        = 42
N_TREES     = 200

raw = pd.read_csv(INPUT_FILE)
print("Archivo:", INPUT_FILE)
# Duplicados: mismo canal, título y fecha (no se usa video_id porque Excel convierte en #NAME? los que empiezan con "-")
n0 = len(raw)
claves = [c for c in ["channel_id", "title", "published_at"] if c in raw.columns]
raw = raw.drop_duplicates(claves).reset_index(drop=True)
print(f"Videos duplicados eliminados: {n0 - len(raw)}")

# Canales secundarios de un creador que ya está en el corpus (el canal es la unidad de agrupamiento)
EXCLUIR_CANALES = ["SEGUN EL GEORGE", "Cultura VJ"]
if "channel_title" in raw.columns:
    raw = raw[~raw["channel_title"].isin(EXCLUIR_CANALES)].reset_index(drop=True)
    print("Canales excluidos por ser del mismo creador:", EXCLUIR_CANALES)
print(f"Filas: {len(raw):,}")
print("Columnas del CSV:", raw.columns.tolist())

# Detecta la columna del canal. Si no la encuentra, escribe su nombre en CHANNEL_COL.
CHANNEL_COL = None
for c in ["channel_id", "channelId", "channel_title", "channelTitle", "channel_name", "channel", "canal"]:
    if c in raw.columns:
        CHANNEL_COL = c
        break
assert CHANNEL_COL is not None, "No encontré la columna del canal: revisa la lista de columnas y asigna CHANNEL_COL a mano."
print(f"\nColumna de canal: {CHANNEL_COL}")
print(raw[CHANNEL_COL].value_counts())

In [ ]:
# CELDA 2 — Idioma por canal y filtro solo español
# langdetect falla con títulos cortos en MAYÚSCULAS, con emojis o con nombres de juegos en inglés.
# Por eso el idioma se decide por canal con la proporción de palabras vacías del español frente a las del inglés
# en todos sus títulos: un canal es "es" si más de la mitad de las palabras vacías encontradas son españolas.
STOP_ES = set("de la el en y que los las un una por con para del es se no su al lo más como ya ser este esta nuevo nueva todo sin muy qué así ha".split())
STOP_EN = set("the of and to in is for on with you this that it are be at your new how why what was not all from best can will".split())

def _stop(t, S):
    return sum(w in S for w in re.findall(r"[a-záéíóúñü]+", str(t).lower()))

extra = [c for c in ["channel_title", "published_at", "duration_seconds", "extraido_el"] if c in raw.columns and c != CHANNEL_COL]
df = raw[["title", "views", CHANNEL_COL] + extra].dropna(subset=["title", "views"]).copy()
df["views"] = pd.to_numeric(df["views"], errors="coerce")
df = df.dropna(subset=["views"]).reset_index(drop=True)
df["views"] = df["views"].astype(int)

df["stop_es"] = df["title"].apply(lambda t: _stop(t, STOP_ES))
df["stop_en"] = df["title"].apply(lambda t: _stop(t, STOP_EN))
resumen = (df.groupby(CHANNEL_COL)
             .agg(videos=("title", "size"), stop_es=("stop_es", "sum"), stop_en=("stop_en", "sum"))
             .sort_values("videos", ascending=False))
resumen["prop_es"] = (resumen["stop_es"] / (resumen["stop_es"] + resumen["stop_en"])).round(2)
resumen["idioma"] = np.where(resumen["prop_es"] > 0.5, "es", "en")
if "channel_title" in df.columns:
    resumen.insert(0, "nombre", df.groupby(CHANNEL_COL)["channel_title"].first())
print(resumen.to_string())

canales_es = resumen.index[resumen["idioma"] == "es"].tolist()
df = df[df[CHANNEL_COL].isin(canales_es)].drop(columns=["stop_es", "stop_en"]).reset_index(drop=True)
print(f"\nSOLO ESPAÑOL: {len(df):,} videos de {len(canales_es)} canales")
resumen.to_csv("tabla_canales_idioma.csv")

In [ ]:
# CELDA 3 — Rasgos del título y variable objetivo
import emoji

_cb = [
    r'\b(top\s*\d+|\d+\s*(razones|tips|formas|ways|errores))',
    r'\b(increíble|impresionante|shocking|nunca.*antes|por\s*fin)',
    r'\?.*\?', r'!!',
]
def _feats(title):
    t = str(title); w = t.split(); n = len(w)
    nm = sum(1 for x in w if x.isupper() and len(x) > 1)
    nn = len(re.findall(r'\b\d+\b', t))
    return pd.Series({
        "n_palabras": n, "n_chars": len(t),
        "pct_mayus": round(nm / n, 3) if n > 0 else 0,
        "n_nums": nn, "n_excl": t.count('!'), "n_interr": t.count('?'),
        "tiene_num": int(nn > 0), "tiene_excl": int('!' in t), "tiene_interr": int('?' in t),
        "n_simbolos": len(re.findall(r'[^\w\s,.\-!?¡¿\'"()]', t)),   # antes emoji_count
        "n_emojis": emoji.emoji_count(t),
        "clickbait_score": sum(1 for p in _cb if re.search(p, t.lower())),
    })

df = pd.concat([df, df["title"].apply(_feats)], axis=1)
df["log_views"] = np.log10(df["views"] + 1)                       # SIEMPRE log10
df["log_views_centrado"] = df["log_views"] - df.groupby(CHANNEL_COL)["log_views"].transform("mean")

# Antigüedad del video a la fecha de extracción (variable de control, sección 7.4)
# Fecha de la extracción: columna extraido_el si existe; si no, la fecha del nombre del archivo
_m = re.search(r"(\d{8})_\d{6}", INPUT_FILE)
FECHA_CORTE = (pd.to_datetime(df["extraido_el"], utc=True).max() if "extraido_el" in df.columns
               else pd.Timestamp(_m.group(1), tz="UTC") if _m else pd.to_datetime(df["published_at"], utc=True).max())
print("Fecha de corte para la antigüedad:", FECHA_CORTE.date())
if "published_at" in df.columns:
    publicado = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
    df["dias"] = (FECHA_CORTE - publicado).dt.days.clip(lower=0)
    df["antiguedad"] = np.log10(df["dias"] + 1)

ESTRUCTURA = ["n_palabras", "n_chars", "pct_mayus", "n_nums", "n_excl", "n_interr",
              "tiene_num", "tiene_excl", "tiene_interr"]
ESTILO     = ["n_simbolos", "n_emojis", "clickbait_score"]
print(df[ESTRUCTURA + ESTILO].describe().T.round(3))

In [ ]:
# CELDA 4 — Sentimiento: VADER (línea base) y pysentimiento en español (usa GPU)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from pysentimiento import create_analyzer

vader = SentimentIntensityAnalyzer()
sc = df["title"].apply(lambda t: vader.polarity_scores(str(t)))
for k in ["compound", "pos", "neg", "neu"]:
    df[f"vader_{k}"] = sc.apply(lambda s: s[k])
df["vader_label"] = np.where(df["vader_compound"] >= 0.05, "POS",
                     np.where(df["vader_compound"] <= -0.05, "NEG", "NEU"))

analyzer = create_analyzer(task="sentiment", lang="es")
salidas = analyzer.predict(df["title"].astype(str).tolist())
df["ps_pos"] = [o.probas["POS"] for o in salidas]
df["ps_neu"] = [o.probas["NEU"] for o in salidas]
df["ps_neg"] = [o.probas["NEG"] for o in salidas]
df["ps_label"] = [o.output for o in salidas]

VADER  = ["vader_compound", "vader_pos", "vader_neg", "vader_neu"]
PYSENT = ["ps_pos", "ps_neu", "ps_neg"]

print("Distribución VADER (%):\n", (df["vader_label"].value_counts(normalize=True) * 100).round(1))
print("\nDistribución pysentimiento (%):\n", (df["ps_label"].value_counts(normalize=True) * 100).round(1))
print("\nCoincidencia VADER vs pysentimiento:\n", pd.crosstab(df["vader_label"], df["ps_label"]))

# Diagnóstico: VADER trata "no" (español) como palabra negativa (valencia -1.2)
tiene_no = df["title"].str.contains(r"\bno\b", case=False, regex=True)
neg_v = df["vader_label"] == "NEG"
print(f"\nTítulos que VADER marca negativos: {neg_v.sum()}")
print(f"  de ellos contienen la palabra 'no': {(neg_v & tiene_no).sum()} ({(neg_v & tiene_no).sum() / max(neg_v.sum(), 1):.1%})")
df.to_csv("titulos_solo_es_con_rasgos.csv", index=False)
print(f"\nGuardado titulos_solo_es_con_rasgos.csv: {len(df):,} videos de {df[CHANNEL_COL].nunique()} canales")
try:
    from google.colab import files
    files.download("titulos_solo_es_con_rasgos.csv")   # respaldo: con este archivo no hay que repetir la celda 4
except Exception:
    pass

In [ ]:
# CELDA 4b — Configuración común de los experimentos (córrela antes de cualquier celda de la 5 a la 11)
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, RepeatedKFold, cross_val_score, cross_validate, permutation_test_score

def rf():
    return RandomForestRegressor(n_estimators=N_TREES, random_state=SEED, n_jobs=-1)

CONFIGS = {
    "C1 estructura":              ESTRUCTURA,
    "C2 + estilo visual":         ESTRUCTURA + ESTILO,
    "C3 + sentimiento VADER":     ESTRUCTURA + ESTILO + VADER,
    "C4 + sentimiento español":   ESTRUCTURA + ESTILO + PYSENT,
}
y = df["log_views"].values
mejor = "C4 + sentimiento español"   # la celda 8 lo actualiza con la configuración de mayor R2
print("Listo:", len(df), "videos |", df[CHANNEL_COL].nunique(), "canales | configuraciones:", list(CONFIGS))

In [ ]:
# CELDA 5 — E1: estadísticos de vistas (Tabla 8) e histograma
v = df["views"]
tabla4 = pd.DataFrame({
    "vistas": [v.min(), v.median(), v.mean(), v.max(), v.std()],
    "log10(vistas+1)": [df["log_views"].min(), df["log_views"].median(), df["log_views"].mean(),
                        df["log_views"].max(), df["log_views"].std()],
}, index=["Mínimo", "Mediana", "Media", "Máximo", "Desv. estándar"])
print(tabla4.round(2).to_string())
print(f"\nMedia / mediana = {v.mean() / v.median():.1f}")
tabla4.to_csv("tabla8_descriptivos.csv")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(v[v <= v.quantile(.95)], bins=50, color="#2E5496"); ax[0].set_title("Vistas (hasta P95)")
ax[1].hist(df["log_views"], bins=50, color="#2E5496"); ax[1].set_title("log10(vistas + 1)")
plt.tight_layout(); plt.savefig("fig_distribucion_vistas.png", dpi=200); plt.show()

In [ ]:
# CELDA 6 — E2: sentimiento vs. vistas (Kruskal-Wallis)
for col in ["vader_label", "ps_label"]:
    g = df.groupby(col)["log_views"]
    print(f"\n{col}:\n", g.agg(["count", "mean", "median"]).round(3))
    grupos = [x.values for _, x in g]
    h, p = stats.kruskal(*grupos)
    print(f"  Kruskal-Wallis H = {h:.2f}, p = {p:.4f}")

In [ ]:
# CELDA 7 — E3: dependencia (MI, Spearman, Kendall, dCor) con pruebas de permutación y BH
import dcor
from sklearn.feature_selection import mutual_info_regression

FEATS = ESTRUCTURA + ESTILO + PYSENT + VADER
N_PERM = 1000
rng = np.random.default_rng(SEED)
y = df["log_views"].values
X_all = df[FEATS].fillna(0)
mi_obs = mutual_info_regression(X_all, y, random_state=SEED)      # en NATS

def bh(p):
    p = np.nan_to_num(np.asarray(p, dtype=float), nan=1.0); o = np.argsort(p); m = len(p)
    q = p[o] * m / np.arange(1, m + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(m); out[o] = np.clip(q, 0, 1); return out

import time
filas, t0 = [], time.time()
for j, f in enumerate(FEATS):
    x = X_all[f].values.astype(float)
    rho, p_sp = stats.spearmanr(x, y)
    tau, _ = stats.kendalltau(x, y)
    nulas = [mutual_info_regression(x.reshape(-1, 1), rng.permutation(y), random_state=SEED)[0] for _ in range(N_PERM)]
    p_mi = (1 + np.sum(np.array(nulas) >= mi_obs[j])) / (N_PERM + 1)
    xs, ys = x, y
    dc = dcor.distance_correlation(xs, ys)
    p_dc = dcor.independence.distance_covariance_test(xs, ys, num_resamples=N_PERM, random_state=SEED).pvalue
    filas.append(dict(rasgo=f, MI_nats=mi_obs[j], p_MI=p_mi, rho=rho, p_rho=p_sp, tau=tau, dCor=dc, p_dCor=p_dc))
    print(f"  avance: {j + 1}/{len(FEATS)} rasgos ({f}) | {(time.time() - t0) / 60:.0f} min")

dep = pd.DataFrame(filas)
for c in ["p_MI", "p_rho", "p_dCor"]:
    dep[c.replace("p_", "q_")] = bh(dep[c])
dep = dep.sort_values("MI_nats", ascending=False)
print(dep.round(4).to_string(index=False))
dep.to_csv("tabla11_dependencia.csv", index=False)

In [ ]:
# CELDA 8 — Ablación C1–C4 con KFold repetido y prueba t corregida (H1, H2)
K, R = 5, 5
rkf = RepeatedKFold(n_splits=K, n_repeats=R, random_state=SEED)
y = df["log_views"].values
res, r2s = [], {}
for nombre, cols in CONFIGS.items():
    cv = cross_validate(rf(), df[cols].fillna(0).values, y, cv=rkf,
                        scoring=("r2", "neg_mean_absolute_error"), n_jobs=-1)
    r2s[nombre] = cv["test_r2"]
    res.append(dict(config=nombre, n_rasgos=len(cols), R2=cv["test_r2"].mean(), R2_de=cv["test_r2"].std(),
                    MAE_log10=-cv["test_neg_mean_absolute_error"].mean()))
abl = pd.DataFrame(res)
print(abl.round(4).to_string(index=False))

def t_corregida(a, b, k=K, r=R):
    d = np.asarray(b) - np.asarray(a)
    t = d.mean() / np.sqrt((1 / (k * r) + 1 / (k - 1)) * d.var(ddof=1))
    return d.mean(), t, 1 - stats.t.cdf(t, df=k * r - 1)

print("\nComparaciones (Nadeau y Bengio):")
for a, b in [("C1 estructura", "C2 + estilo visual"), ("C2 + estilo visual", "C3 + sentimiento VADER"),
             ("C2 + estilo visual", "C4 + sentimiento español"), ("C3 + sentimiento VADER", "C4 + sentimiento español")]:
    dm, t, p = t_corregida(r2s[a], r2s[b])
    print(f"  {a} -> {b}: dR2 = {dm:+.4f}, t = {t:.2f}, p = {p:.4f}")
abl.to_csv("tabla12_ablacion.csv", index=False)

mejor = abl.sort_values("R2", ascending=False)["config"].iloc[0]
print(f"\nMejor configuración: {mejor}. La prueba de permutación de H1 se corre en la celda 13.")

In [ ]:
# CELDA 9 — CF1: estabilidad 5 vs. 10 particiones
X_best = df[CONFIGS[mejor]].fillna(0).values
for k in [5, 10]:
    r2 = cross_val_score(rf(), X_best, y, cv=KFold(k, shuffle=True, random_state=SEED), scoring="r2", n_jobs=-1)
    print(f"{k}-fold: R2 = {r2.mean():.4f} ± {r2.std():.4f}")

In [ ]:
# CELDA 10 — CF2: validación dejando un canal fuera (H3)
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
logo = LeaveOneGroupOut()
groups = df[CHANNEL_COL].values
y_c = df["log_views_centrado"].values
cf2 = []
for nombre, cols in CONFIGS.items():
    X = df[cols].fillna(0).values
    r2_rand = cross_val_score(rf(), X, y, cv=KFold(5, shuffle=True, random_state=SEED), scoring="r2", n_jobs=-1).mean()
    pred = cross_val_predict(rf(), X, y, cv=logo, groups=groups, n_jobs=-1)
    pred_c = cross_val_predict(rf(), X, y_c, cv=logo, groups=groups, n_jobs=-1)
    cf2.append(dict(config=nombre, R2_aleatorio=r2_rand,
                    R2_canal_fuera=r2_score(y, pred), MAE_canal_fuera=mean_absolute_error(y, pred),
                    R2_centrado_canal_fuera=r2_score(y_c, pred_c)))
cf2 = pd.DataFrame(cf2)
cf2["fuga_por_canal"] = cf2["R2_aleatorio"] - cf2["R2_canal_fuera"]
print(f"{df[CHANNEL_COL].nunique()} canales\n")
print(cf2.round(4).to_string(index=False))
cf2.to_csv("tabla13_canal_fuera.csv", index=False)

# Robustez: árboles regularizados (sección 7.3)
X4 = df[CONFIGS["C4 + sentimiento español"]].fillna(0).values
reg = []
for nombre, kw in [("sin restricción", {}), ("max_depth=6", dict(max_depth=6)),
                   ("min_samples_leaf=15", dict(min_samples_leaf=15)),
                   ("max_depth=6 y min_samples_leaf=15", dict(max_depth=6, min_samples_leaf=15))]:
    m = lambda: RandomForestRegressor(n_estimators=N_TREES, random_state=SEED, n_jobs=-1, **kw)
    reg.append(dict(arboles=nombre,
                    R2_aleatorio=cross_val_score(m(), X4, y, cv=KFold(5, shuffle=True, random_state=SEED), scoring="r2").mean(),
                    R2_canal_fuera=r2_score(y, cross_val_predict(m(), X4, y, cv=logo, groups=groups)),
                    R2_canal_fuera_centrado=r2_score(y_c, cross_val_predict(m(), X4, y_c, cv=logo, groups=groups))))
reg = pd.DataFrame(reg); print(reg.round(3).to_string(index=False)); reg.to_csv("tabla14_regularizacion.csv", index=False)

In [ ]:
# CELDA 10b — CF2b: el título dentro del canal y desempeño por canal (secciones 7.4 y 7.6)
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
ch = CHANNEL_COL
rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=SEED)
def t_corr(a, b, k=5, r=5):
    dd = np.asarray(b) - np.asarray(a)
    t = dd.mean() / np.sqrt((1 / (k * r) + 1 / (k - 1)) * dd.var(ddof=1))
    return dd.mean(), 1 - stats.t.cdf(t, df=k * r - 1)

# 1) Varianza explicada por el canal
gm = df.groupby(ch)["log_views"].transform("mean")
eta2 = ((gm - y.mean())**2).sum() / ((y - y.mean())**2).sum()
m0 = smf.mixedlm("log_views ~ 1", df, groups=df[ch]).fit()
icc = float(m0.cov_re.iloc[0, 0]) / (float(m0.cov_re.iloc[0, 0]) + m0.scale)
print(f"eta2 canal = {eta2:.3f} | ICC = {icc:.3f}")

# 2) Solo canal vs canal + título (C4)
X_ch = pd.get_dummies(df[ch]).astype(float).values
X_t = df[CONFIGS["C4 + sentimiento español"]].fillna(0).values
s_t = cross_val_score(rf(), X_t, y, cv=rkf, scoring="r2", n_jobs=-1)
s_c = cross_val_score(rf(), X_ch, y, cv=rkf, scoring="r2", n_jobs=-1)
s_ct = cross_val_score(rf(), np.hstack([X_ch, X_t]), y, cv=rkf, scoring="r2", n_jobs=-1)
print(f"Solo título {s_t.mean():.3f} ± {s_t.std():.3f} | Solo canal {s_c.mean():.3f} ± {s_c.std():.3f} | Canal + título {s_ct.mean():.3f} ± {s_ct.std():.3f}")
print("  canal -> canal+título: dR2 = %+.4f, p = %.4f" % t_corr(s_c, s_ct))

# 3) ¿El título identifica al canal?
acc = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1), X_t, df[ch],
                      cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="accuracy")
print(f"Exactitud al identificar el canal: {acc.mean():.3f} (clase mayoritaria {df[ch].value_counts(normalize=True).max():.3f})")

# 4) Dentro del canal: rasgos y objetivo centrados
cols4 = CONFIGS["C4 + sentimiento español"]
Xc = df[cols4].fillna(0) - df.groupby(ch)[cols4].transform("mean")
s_w = cross_val_score(rf(), Xc.values, df["log_views_centrado"].values, cv=rkf, scoring="r2", n_jobs=-1)
print(f"R2 dentro del canal (centrado): {s_w.mean():.3f} ± {s_w.std():.3f}")

# 5) Modelo mixto con rasgos estandarizados
feats = ["n_palabras","n_chars","pct_mayus","n_nums","n_excl","n_interr","n_simbolos","n_emojis","clickbait_score","ps_pos","ps_neg"]
dz = df.copy()
for f in feats: dz[f] = (dz[f] - dz[f].mean()) / (dz[f].std(ddof=0) or 1)
mm = smf.mixedlm("log_views ~ " + " + ".join(feats), dz, groups=dz[ch]).fit(reml=True)
fe = np.asarray(mm.model.exog) @ mm.fe_params.values
vf, vr, ve = fe.var(), float(mm.cov_re.iloc[0, 0]), mm.scale
tab = pd.DataFrame({"coef": mm.fe_params, "p": mm.pvalues[mm.fe_params.index]}).drop("Intercept")
tab["q"] = multipletests(tab["p"], method="fdr_bh")[1]
print(tab.round(4).sort_values("p").to_string())
print(f"R2 marginal = {vf/(vf+vr+ve):.4f} | R2 condicional = {(vf+vr)/(vf+vr+ve):.4f}")

# 6) Spearman global vs dentro del canal
yc = df["log_views_centrado"]
filas = []
for f in ESTRUCTURA + ESTILO + VADER + PYSENT:
    rg, _ = stats.spearmanr(df[f], y)
    rd, pd_ = stats.spearmanr(df[f] - df.groupby(ch)[f].transform("mean"), yc)
    filas.append(dict(rasgo=f, rho_global=rg, rho_dentro=rd, p_dentro=pd_))
sp = pd.DataFrame(filas); sp["q_dentro"] = multipletests(np.nan_to_num(sp["p_dentro"], nan=1), method="fdr_bh")[1]
print(sp.round(4).sort_values("p_dentro").to_string(index=False)); sp.to_csv("tabla16_dentro_canal.csv", index=False)

# 7) Desempeño por canal dejando el canal fuera (C4)
pred = cross_val_predict(rf(), X_t, y, cv=LeaveOneGroupOut(), groups=df[ch], n_jobs=-1)
pc = (df.assign(pred=pred, err=np.abs(pred - y)).groupby(ch)
        .agg(videos=("title", "size"), media_real=("log_views", "mean"), media_pred=("pred", "mean"), MAE=("err", "mean")))
pc["sesgo"] = pc["media_pred"] - pc["media_real"]
print(pc.sort_values("MAE").round(2).to_string()); pc.to_csv("tabla19_por_canal.csv")

# 8) Control por antigüedad del video
if "antiguedad" in df.columns:
    da = df.dropna(subset=["antiguedad"]).copy()
    cen = lambda c: da[c] - da.groupby(ch)[c].transform("mean")
    a_c, n_c, y_c2 = cen("antiguedad"), cen("ps_neg"), cen("log_views")
    print("Spearman dentro del canal antigüedad-vistas:", round(stats.spearmanr(a_c, y_c2)[0], 3))
    # correlación parcial de Spearman ps_neg-vistas controlando antigüedad (residuos sobre rangos)
    from scipy.stats import rankdata
    ra, rn, ry = rankdata(a_c), rankdata(n_c), rankdata(y_c2)
    res = lambda v: v - np.polyval(np.polyfit(ra, v, 1), ra)
    rp, pp = stats.pearsonr(res(rn), res(ry))
    print(f"ps_neg parcial dentro del canal (controlando antigüedad): rho = {rp:.3f}, p = {pp:.4f}")
    dz2 = da.copy()
    for f in feats + ["antiguedad"]: dz2[f] = (dz2[f] - dz2[f].mean()) / (dz2[f].std(ddof=0) or 1)
    mm2 = smf.mixedlm("log_views ~ antiguedad + " + " + ".join(feats), dz2, groups=dz2[ch]).fit(reml=True)
    print(mm2.summary().tables[1].loc[["antiguedad", "ps_neg"]])
    Xch2 = pd.get_dummies(da[ch]).astype(float).values; ya = da["log_views"].values
    Xt2 = da[cols4].fillna(0).values; Xa = da[["antiguedad"]].values
    s_ca = cross_val_score(rf(), np.hstack([Xch2, Xa]), ya, cv=rkf, scoring="r2", n_jobs=-1)
    s_cat = cross_val_score(rf(), np.hstack([Xch2, Xa, Xt2]), ya, cv=rkf, scoring="r2", n_jobs=-1)
    print(f"Canal + antigüedad {s_ca.mean():.3f} | + título {s_cat.mean():.3f}")
    print("  título sobre canal + antigüedad: dR2 = %+.4f, p = %.4f" % t_corr(s_ca, s_cat))

# 9) Sin el canal C (el de mayor audiencia) y rasgos centrados evaluados dejando un canal fuera
grande = df.groupby(ch)["log_views"].mean().idxmax()
d2 = df[df[ch] != grande].copy()
d2["log_views_centrado"] = d2["log_views"] - d2.groupby(ch)["log_views"].transform("mean")
y2 = d2["log_views"].values
print(f"Sin {grande}: R2 aleatorio C4 = {cross_val_score(rf(), d2[cols4].fillna(0).values, y2, cv=rkf, scoring='r2', n_jobs=-1).mean():.3f}"
      f" | R2 canal fuera = {r2_score(y2, cross_val_predict(rf(), d2[cols4].fillna(0).values, y2, cv=LeaveOneGroupOut(), groups=d2[ch], n_jobs=-1)):.3f}")
pred_xc = cross_val_predict(rf(), Xc.values, df["log_views_centrado"].values, cv=LeaveOneGroupOut(), groups=df[ch], n_jobs=-1)
print(f"Rasgos y objetivo centrados, dejando un canal fuera: R2 = {r2_score(df['log_views_centrado'].values, pred_xc):.3f}")

In [ ]:
# CELDA 11 — CF3: importancia por permutación y SHAP sobre datos de PRUEBA
import shap
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
cols = CONFIGS[mejor]
X = df[cols].fillna(0)
imp_folds, sv_all, Xte_all = [], [], []
for tr, te in KFold(5, shuffle=True, random_state=SEED).split(X):
    m = rf().fit(X.iloc[tr], y[tr])
    pi = permutation_importance(m, X.iloc[te], y[te], n_repeats=20, random_state=SEED, scoring="r2", n_jobs=-1)
    imp_folds.append(pi.importances_mean)
    sv_all.append(shap.TreeExplainer(m).shap_values(X.iloc[te])); Xte_all.append(X.iloc[te])
imp = pd.DataFrame({"rasgo": cols, "perm_media": np.mean(imp_folds, 0), "perm_de": np.std(imp_folds, 0),
                    "shap_abs_medio": np.abs(np.vstack(sv_all)).mean(0)}).sort_values("perm_media", ascending=False)
print(imp.round(4).to_string(index=False))
imp.to_csv("tabla17_importancia_test.csv", index=False)

# Importancia agrupada por concepto (rasgos correlacionados se permutan juntos)
GRUPOS = {"Longitud": ["n_palabras", "n_chars"], "Mayúsculas": ["pct_mayus"], "Números": ["n_nums", "tiene_num"],
          "Exclamación": ["n_excl", "tiene_excl"], "Interrogación": ["n_interr", "tiene_interr"],
          "Símbolos y emojis": ["n_simbolos", "n_emojis"], "Clickbait": ["clickbait_score"],
          "Sentimiento (pysentimiento)": ["ps_pos", "ps_neu", "ps_neg"],
          "Sentimiento (VADER)": ["vader_compound", "vader_pos", "vader_neg", "vader_neu"]}
GRUPOS = {k: [c for c in v if c in cols] for k, v in GRUPOS.items()}
GRUPOS = {k: v for k, v in GRUPOS.items() if v}   # solo los grupos presentes en la configuración evaluada
rng_g = np.random.default_rng(SEED); caidas = {k: [] for k in GRUPOS}
for tr, te in KFold(5, shuffle=True, random_state=SEED).split(X):
    m = rf().fit(X.iloc[tr], y[tr]); Xte = X.iloc[te]; base = r2_score(y[te], m.predict(Xte))
    for k, cg in GRUPOS.items():
        dd = []
        for _ in range(20):
            Xp = Xte.copy(); Xp[cg] = Xte[cg].values[rng_g.permutation(len(Xte))]
            dd.append(base - r2_score(y[te], m.predict(Xp)))
        caidas[k].append(np.mean(dd))
grp = pd.DataFrame({"grupo": list(caidas), "caida_R2": [np.mean(v) for v in caidas.values()],
                    "de": [np.std(v) for v in caidas.values()]}).sort_values("caida_R2", ascending=False)
print(grp.round(3).to_string(index=False)); grp.to_csv("tabla18_importancia_agrupada.csv", index=False)
shap.summary_plot(np.vstack(sv_all), pd.concat(Xte_all), show=False, max_display=15)
plt.title("SHAP sobre datos de prueba"); plt.tight_layout()
plt.savefig("fig_shap_test.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# CELDA 12 — Resumen de resultados y descarga de resultados
print("=" * 60)
print(f"Corpus: {len(df):,} videos | {df[CHANNEL_COL].nunique()} canales en español")
print(f"Vistas: mediana {df['views'].median():,.0f} | media {df['views'].mean():,.0f}")
print(f"Neutros VADER: {(df['vader_label']=='NEU').mean():.1%} | neutros pysentimiento: {(df['ps_label']=='NEU').mean():.1%}")
print("\nAblación:\n", abl.round(4).to_string(index=False))
print("\nCF2:\n", cf2.round(4).to_string(index=False))
print("=" * 60)
import shutil
salidas = [f for f in os.listdir(".") if f.endswith((".csv", ".png")) and not f.startswith("youtube_videos")]
os.makedirs("resultados_solo_es", exist_ok=True)
for f in salidas: shutil.copy(f, "resultados_solo_es")
shutil.make_archive("resultados_solo_es", "zip", "resultados_solo_es")
files.download("resultados_solo_es.zip")

In [ ]:
# CELDA 13 — H1: ¿el mejor modelo supera al azar? Prueba de permutación con avance visible
# Es la celda más lenta (entrena 5 modelos por permutación); por eso va después de la descarga de la celda 12.
import time
N_H1 = 1000
X_h1 = df[CONFIGS[mejor]].fillna(0).values
cv_h1 = KFold(5, shuffle=True, random_state=SEED)
y_h1 = df["log_views"].values
r2_obs = cross_val_score(rf(), X_h1, y_h1, cv=cv_h1, scoring="r2", n_jobs=-1).mean()
rng_h1 = np.random.default_rng(SEED); r2_nulos = []; t0 = time.time()
for i in range(N_H1):
    r2_nulos.append(cross_val_score(rf(), X_h1, rng_h1.permutation(y_h1), cv=cv_h1, scoring="r2", n_jobs=-1).mean())
    if (i + 1) % 25 == 0:
        p_h1 = (1 + sum(v >= r2_obs for v in r2_nulos)) / (len(r2_nulos) + 1)
        print(f"  {i + 1}/{N_H1} | {(time.time() - t0) / 60:.0f} min | R2 = {r2_obs:.4f} | nulo medio = {np.mean(r2_nulos):.4f} | nulo máx = {max(r2_nulos):.4f} | p = {p_h1:.4f}")
        pd.DataFrame({"r2_nulo": r2_nulos}).assign(r2_observado=r2_obs).to_csv("h1_permutacion.csv", index=False)
p_h1 = (1 + sum(v >= r2_obs for v in r2_nulos)) / (N_H1 + 1)
print(f"\nH1 ({mejor}): R2 = {r2_obs:.4f}, R2 nulo medio = {np.mean(r2_nulos):.4f}, p = {p_h1:.4f} ({N_H1} permutaciones)")
try:
    from google.colab import files
    files.download("h1_permutacion.csv")
except Exception:
    pass